In [38]:
# 파이썬에서 파일을 분리하는 전처리 코드
import pandas as pd

file_name = 'busan_bus_stops.csv'

# 데이터 불러오기 (한글 인코딩 처리)
try:
    df = pd.read_csv(file_name, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_name, encoding='cp949')

df.head(5)
df.info

<bound method DataFrame.info of              정류장번호        정류장명         위도          경도       정보수집일  모바일단축번호  \
0     BSB192080301       강서구청역  35.211314  128.979613  2025-10-31  12298.0   
1     BSB192080302       강서구청역  35.211185  128.980946  2025-10-31  92223.0   
2     BSB192080701       강서구청역  35.210875  128.980423  2025-10-31  92390.0   
3     BSB192090101       부산교도소  35.215958  128.951537  2025-10-31  92298.0   
4     BSB192090102     중리1구꽃동네  35.218346  128.955439  2025-10-31  92299.0   
...            ...         ...        ...         ...         ...      ...   
9970  GGB277101222  회동교차로(미정차)  35.234100  129.115767  2025-10-31      0.0   
9971  GGB277101223   원동IC(미정차)  35.199383  129.114433  2025-10-31      0.0   
9972  GGB277101224   원동IC(미정차)  35.199267  129.114733  2025-10-31      0.0   
9973  GGB277103823   장안TG(미정차)  35.328833  129.241567  2025-10-31      0.0   
9974  GGB277103824   장안TG(미정차)  35.329083  129.241433  2025-10-31      0.0   

      도시코드    도시명  관리도시명  
0   

In [45]:
file_path = '서구_장기요양기관내역.csv'
df = pd.read_csv(file_path, header=2, encoding='utf-8')

cols_to_drop = ['평가결과', '방문목욕차량','잔여','대기']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

df.head(5)

,연번,장기요양기관,급여종류,정원,현원,주소,전화번호
0,1,인창서구재가노인복지센터,주야간보호,100,96,"부산광역시 서구 망양로 72-1 (동대신동3가, 보현빌딩)",051-242-0906
1,2,효송주야간보호센터,주야간보호,60,49,부산광역시 서구 암남로14번길 12 (암남동),051-242-7085
2,3,송도노인주간보호센터,주야간보호,57,45,부산광역시 서구 암남공원로 522 6층 (암남동),051-248-3315
3,4,소은노인주간보호센터,주야간보호,25,19,부산광역시 서구 천마로199번길 13 (남부민동),051-248-8888
4,5,서구케어재가노인주간보호센터,주야간보호,93,69,부산광역시 서구 구덕로 249 2~4층 (부용동2가),051-925-9972


In [46]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

file_path = '부산_서구_요양기관내역_전처리.csv'
df = pd.read_csv(file_path, header=2, encoding='utf-8')

df['검색용주소'] = df['주소'].apply(lambda x: str(x).split('(')[0].strip().replace('"', ''))
df['위도'] = None
df['경도'] = None

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)
driver.get("https://www.findlatlng.org/")
time.sleep(3) # 사이트 초기 로딩 대기

xpath_search = '//*[@id="__nuxt"]/div/main/div/div[3]/div[2]/div/div/input'
xpath_lat = '//*[@id="__nuxt"]/div/main/div/div[3]/div[4]/div[2]/span[2]'
xpath_lng = '//*[@id="__nuxt"]/div/main/div/div[3]/div[4]/div[3]/span[2]'

######################################검색
for index, row in df.iterrows():
    address = row['검색용주소']
    
    search_box = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, xpath_search))
    )
    search_box.send_keys(Keys.CONTROL + "a")
    search_box.send_keys(Keys.BACKSPACE)
    time.sleep(0.5)
    search_box.send_keys(address)
    search_box.send_keys(Keys.ENTER)
    
    time.sleep(2)

    lat = driver.find_element(By.XPATH, xpath_lat).text
    lng = driver.find_element(By.XPATH, xpath_lng).text

    df.at[index, '위도'] = lat
    df.at[index, '경도'] = lng
    print(f"✅ [{index+1}/{len(df)}] {address} -> 위도: {lat}, 경도: {lng}")
driver.quit()

df = df.drop(columns=['검색용주소'])
final_filename = 'seogu_centers_crawled.csv'
df.to_csv(final_filename, index=False, encoding='utf-8-sig')

총 61개의 기관 데이터를 크롤링합니다...

✅ [1/61] 부산광역시 서구 망양로  72-1 -> 위도: 35.1154261584694, 경도: 129.016953991395
✅ [2/61] 부산광역시 서구 암남로14번길  12 -> 위도: 35.0792582317386, 경도: 129.015504340039
✅ [3/61] 부산광역시 서구 암남공원로  522 6층 -> 위도: 35.0797369885957, 경도: 129.012137977952
✅ [4/61] 부산광역시 서구 천마로199번길  13 -> 위도: 35.0931264367831, 경도: 129.021308688741
✅ [5/61] 부산광역시 서구 구덕로  249 2~4층 -> 위도: 35.0931264367831, 경도: 129.021308688741
✅ [6/61] 부산광역시 서구 구덕로  164-2 3층 -> 위도: 35.0991167718877, 경도: 129.020069271539
✅ [7/61] 부산광역시 서구 암남공원로  522 신관동1층 -> 위도: 35.0797369885957, 경도: 129.012137977952
✅ [8/61] 부산광역시 서구 구덕로  284-1 3층 -> 위도: 35.1094916821085, 경도: 129.018402142852
✅ [9/61] 부산광역시 서구 구덕로286번길  3 302,303호 -> 위도: 35.1094916821085, 경도: 129.018402142852
✅ [10/61] 부산광역시 서구 구덕로  350 2층10호 -> 위도: 35.1150726941653, 경도: 129.01590480137
✅ [11/61] 부산광역시 서구 꽃마을로  48 301동1층102호 -> 위도: 35.1183783315399, 경도: 129.012199836572
✅ [12/61] 부산광역시 서구 해돋이로  203 1층 -> 위도: 35.0951823956857, 경도: 129.018712565914
✅ [13/61] 부산광역시 서구 천마로  189

In [48]:
cols_to_drop = ['잔여','대기']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')
df

,연번,장기요양기관,급여종류,정원,현원,주소,전화번호,위도,경도
0,1,인창서구재가노인복지센터,주야간보호,100,96,"부산광역시 서구 망양로 72-1 (동대신동3가, 보현빌딩)",051-242-0906,35.1154261584694,129.016953991395
1,2,효송주야간보호센터,주야간보호,60,49,부산광역시 서구 암남로14번길 12 (암남동),051-242-7085,35.0792582317386,129.015504340039
2,3,송도노인주간보호센터,주야간보호,57,45,부산광역시 서구 암남공원로 522 6층 (암남동),051-248-3315,35.0797369885957,129.012137977952
3,4,소은노인주간보호센터,주야간보호,25,19,부산광역시 서구 천마로199번길 13 (남부민동),051-248-8888,35.0931264367831,129.021308688741
4,5,서구케어재가노인주간보호센터,주야간보호,93,69,부산광역시 서구 구덕로 249 2~4층 (부용동2가),051-925-9972,35.0931264367831,129.021308688741
...,...,...,...,...,...,...,...,...,...
56,57,(주)가온누리종합복지센터,복지용구,-,-,"부산광역시 서구 구덕로286번길 3 302,303호 (동대신동1가)",051-242-7355,35.1150726941653,129.01590480137
57,58,동진의료기,복지용구,-,-,부산광역시 서구 대신공원로 17 (동대신동3가),051-247-3136,35.1195152655996,129.016156646628
58,59,안나노인건강센터,노인요양시설(개정법),80,62,부산광역시 서구 꽃마을로163번길 94-16 (서대신동3가),051-245-0035,35.1277283699183,129.00478102296
59,60,실버웰요양센터,노인요양시설(개정법),150,46,부산광역시 서구 시약로 35 (서대신동3가),051-242-8050,35.1126337583154,129.008502036752


In [49]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# ==========================================
# 1. 영도구 파일 불러오기 및 정제
# ==========================================
# 새로 업로드하신 파일명 적용
file_path = '영도구_장기요양기관내역.csv'
df = pd.read_csv(file_path, header=2, encoding='utf-8')

# 불필요한 열 제거
cols_to_drop = ['평가결과', '방문목욕차량']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

# 주소 정제 (괄호 및 따옴표 제거)
df = df.dropna(subset=['장기요양기관', '주소'])
df['검색용주소'] = df['주소'].apply(lambda x: str(x).split('(')[0].strip().replace('"', ''))
df['위도'] = None
df['경도'] = None

print(f"총 {len(df)}개의 영도구 기관 데이터를 크롤링합니다...\n")

# ==========================================
# 2. Selenium 크롤러 구동
# ==========================================
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)
driver.get("https://www.findlatlng.org/")
time.sleep(3) # 사이트 초기 로딩 대기

# 확인 완료된 최신 XPath 적용
xpath_search = '//*[@id="__nuxt"]/div/main/div/div[3]/div[2]/div/div/input'
xpath_lat = '//*[@id="__nuxt"]/div/main/div/div[3]/div[4]/div[2]/span[2]' 
xpath_lng = '//*[@id="__nuxt"]/div/main/div/div[3]/div[4]/div[3]/span[2]'

# ==========================================
# 3. 검색 및 좌표 추출 반복문
# ==========================================
for index, row in df.iterrows():
    address = row['검색용주소']
    
    try:
        # 주소 입력창 찾기
        search_box = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, xpath_search))
        )
        
        # 기존 텍스트 전체 선택(Ctrl+A) 후 삭제(Backspace)
        search_box.send_keys(Keys.CONTROL + "a")
        search_box.send_keys(Keys.BACKSPACE)
        time.sleep(0.5)
        
        # 새 주소 입력 후 검색
        search_box.send_keys(address)
        search_box.send_keys(Keys.ENTER)
        
        # 좌표값 계산 대기
        time.sleep(2) 
        
        # 텍스트 추출 (.text)
        lat = driver.find_element(By.XPATH, xpath_lat).text
        lng = driver.find_element(By.XPATH, xpath_lng).text
        
        # 값 저장
        df.at[index, '위도'] = lat
        df.at[index, '경도'] = lng
        
        print(f"✅ [{index+1}/{len(df)}] {address} -> 위도: {lat}, 경도: {lng}")
        
    except Exception as e:
        print(f"❌ [{index+1}/{len(df)}] {address} 검색 실패")

driver.quit()

# ==========================================
# 4. 결과 저장
# ==========================================
df = df.drop(columns=['검색용주소'])

# 영도구 전용 출력 파일명으로 저장
final_filename = 'yeongdogu_care_centers_crawled_final.csv'
df.to_csv(final_filename, index=False, encoding='utf-8-sig')

print(f"\n🎉 크롤링 완료! 영도구 데이터가 '{final_filename}'에 완벽하게 저장되었습니다.")

총 99개의 영도구 기관 데이터를 크롤링합니다...

✅ [1/99] 부산광역시 영도구 와치로  78 -> 위도: 35.0859515078014, 경도: 129.059603122949
✅ [2/99] 부산광역시 영도구 와치로  78 -> 위도: 35.0859515078014, 경도: 129.059603122949
✅ [3/99] 부산광역시 영도구 절영로29번길  14 -> 위도: 35.0913300208287, 경도: 129.040097492103
✅ [4/99] 부산광역시 영도구 중리로  35 101호, 201호 -> 위도: 35.0740264903197, 경도: 129.068034917019
✅ [5/99] 부산광역시 영도구 태종로  417 6층 -> 위도: 35.0903954756835, 경도: 129.066622081691
✅ [6/99] 부산광역시 영도구 꿈나무길  60 3층 -> 위도: 35.0889494658427, 경도: 129.044328985437
✅ [7/99] 부산광역시 영도구 꿈나무길  17 -> 위도: 35.0903495766843, 경도: 129.044481144665
✅ [8/99] 부산광역시 영도구 동삼로108번길  7 -> 위도: 35.0827898657922, 경도: 129.071964692742
✅ [9/99] 부산광역시 영도구 와치로  235 1층 -> 위도: 35.0758736195678, 경도: 129.067432699482
✅ [10/99] 부산광역시 영도구 와치로3번길  20 -> 위도: 35.0909156920625, 경도: 129.058240157645
✅ [11/99] 부산광역시 영도구 꿈나무길  183 2~3층 -> 위도: 35.0909156920625, 경도: 129.058240157645
✅ [12/99] 부산광역시 영도구 태종로  382 2층 -> 위도: 35.0928776133579, 경도: 129.065276476472
✅ [13/99] 부산광역시 영도구 한사랑길  114 -> 위도: 35.08926

In [50]:
cols_to_drop = ['잔여','대기']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')
df

,연번,장기요양기관,급여종류,정원,현원,주소,전화번호,위도,경도
0,1,파랑새노인건강센터,치매전담실나형1실,0,0,부산광역시 영도구 와치로 78 (청학동),051-412-5422,35.0859515078014,129.059603122949
1,2,파랑새노인건강센터,치매전담실나형2실,0,0,부산광역시 영도구 와치로 78 (청학동),051-412-5422,35.0859515078014,129.059603122949
2,3,혜원영도구노인복지센터,주야간보호,19,16,부산광역시 영도구 절영로29번길 14 (대교동2가),051-417-6344,35.0913300208287,129.040097492103
3,4,영도주간보호센터,주야간보호,81,67,"부산광역시 영도구 중리로 35 101호, 201호 (동삼동, 영도빌딩)",051-405-7888,35.0740264903197,129.068034917019
4,5,다사랑장기요양기관,주야간보호,31,27,부산광역시 영도구 태종로 417 6층 (청학동),051-404-0676,35.0903954756835,129.066622081691
...,...,...,...,...,...,...,...,...,...
94,95,신기의료기 영도점,복지용구,-,-,부산광역시 영도구 동삼서로 33 1층 (동삼동),051-915-6564,35.0819194206907,129.069010450811
95,96,드림복지용구,복지용구,-,-,"부산광역시 영도구 남항새싹길 81 7동1층2호 (영선동4가, 영도대원아파트 상가)",051-414-5015,35.0813473704741,129.042873036647
96,97,영도요양원,노인요양시설(개정법),29,29,"부산광역시 영도구 중리로 35 101~401호 (동삼동, 영도빌딩)",051-405-7888,35.0813473704741,129.042873036647
97,98,파랑새노인건강센터,노인요양시설(개정법),123,88,부산광역시 영도구 와치로 78 (청학동),051-412-5422,35.0859515078014,129.059603122949


In [23]:
import pandas as pd
import requests
import folium
from math import radians, cos, sin, asin, sqrt
import re

# ==========================================
# 1. 거리 계산 함수 (Haversine 공식)
# ==========================================
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 
    return c * r * 1000 # 미터(m) 단위 반환

# ==========================================
# 2. 메인 분석 및 지도 생성 함수
# ==========================================
def run_enhanced_analysis(target_address):
    api_key = "40641746de52a507f78f6c31be11e9c9"
    headers = {"Authorization": f"KakaoAK {api_key}"}

    print(f"\n🔍 [{target_address}] 입지 분석을 시작합니다...")

    # [1] 주소 -> 좌표 변환
    geo_res = requests.get("https://dapi.kakao.com/v2/local/search/address.json", headers=headers, params={"query": target_address}).json()
    if not geo_res.get('documents'): 
        print("❌ 주소를 찾을 수 없습니다. 정확한 주소를 다시 입력해주세요.")
        return
    
    c_lat, c_lng = float(geo_res['documents'][0]['y']), float(geo_res['documents'][0]['x'])

    # [2] 상급/대학병원 거리 정보 (API)
    h_info = []
    for q in ["종합병원", "대학병원"]:
        res = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", headers=headers, 
                           params={"query": q, "category_group_code": "HP8", "x": c_lng, "y": c_lat, "radius": 15000, "sort": "distance"}).json()
        found = "정보 없음"
        for d in res.get('documents', []):
            if all(w not in d['place_name'] for w in ['동물', '요양', '한방', '치과', '피부과']):
                dist = int(d['distance'])
                found = f"<b>{d['place_name']}</b> ({f'{dist/1000:.1f}km' if dist >= 1000 else f'{dist}m'})"
                break
        h_info.append(found)

    # [3] 지도 초기화
    m = folium.Map(location=[c_lat, c_lng], zoom_start=16, tiles='CartoDB positron')
    folium.Circle(location=[c_lat, c_lng], radius=500, color='#3498db', fill=True, fill_opacity=0.05, weight=2).add_to(m)
    
    # 중앙 타겟 마커
    center_tooltip = f"<div style='width:250px;'><b>[분석 대상지]</b><br>{target_address}<hr>🚨 종합병원: {h_info[0]}<br>🏥 대학병원: {h_info[1]}</div>"
    folium.CircleMarker([c_lat, c_lng], radius=20, color='red', fill=True, fill_color='red', fill_opacity=0.4, z_index=1000).add_to(m)
    folium.Marker([c_lat, c_lng], icon=folium.Icon(color='black', icon='star', prefix='fa'), tooltip=center_tooltip, z_index=1001).add_to(m)

    # [4] CSV 데이터 추가 (시내버스, 요양시설)
    try:
        # A. 시내버스
        bus_df = pd.read_csv('busan_bus_stops.csv', encoding='utf-8-sig')
        for _, row in bus_df.iterrows():
            if haversine(c_lng, c_lat, row['경도'], row['위도']) <= 500:
                folium.Marker([row['위도'], row['경도']], tooltip=f"🚌 시내버스: {row['정류장명']}",
                              icon=folium.Icon(color='orange', icon='flag')).add_to(m)

        # B. 요양시설 (상세 정보 툴팁 추가)
        care_df = pd.concat([pd.read_csv('seogu_care_centers_crawled_final.csv'), 
                             pd.read_csv('yeongdogu_care_centers_crawled_final.csv')], ignore_index=True)
        care_df['위도'] = pd.to_numeric(care_df['위도'], errors='coerce')
        care_df['경도'] = pd.to_numeric(care_df['경도'], errors='coerce')
        
        for _, row in care_df.dropna(subset=['위도', '경도']).iterrows():
            if haversine(c_lng, c_lat, row['경도'], row['위도']) <= 500:
                # 🌟 요청하신 상세 정보 구성 (주소, 전화번호, 정원, 현원)
                care_tooltip = f"""
                <div style='width:200px; font-size:12px;'>
                    <b>🏥 {row['장기요양기관']}</b><br>
                    📍 주소: {row['주소']}<br>
                    📞 전화: {row.get('전화번호', '정보없음')}<br>
                    👥 정원: {row.get('정원', '-')}명 / 현원: {row.get('현원', '-')}명
                </div>
                """
                folium.Marker(
                    [row['위도'], row['경도']], 
                    tooltip=care_tooltip,
                    icon=folium.Icon(color='purple', icon='heart')
                ).add_to(m)
    except Exception as e:
        print(f"⚠️ CSV 로드 에러: {e}")

    # [5] API 데이터 추가 (병원-피부과제외, 약국, 지하철, 시외버스)
    infra_list = [
        {"q": "시외버스정류장", "color": "darkred", "icon": "bus"},
        {"code": "HP8", "name": "병원", "color": "red", "icon": "plus"},
        {"code": "PM9", "name": "약국", "color": "green", "icon": "medkit"},
        {"code": "SW8", "name": "지하철역", "color": "blue", "icon": "info-sign"}
    ]
    
    for item in infra_list:
        url = "https://dapi.kakao.com/v2/local/search/category.json" if "code" in item else "https://dapi.kakao.com/v2/local/search/keyword.json"
        params = {"x": c_lng, "y": c_lat, "radius": 500, "sort": "distance", "size": 15}
        if "code" in item: params["category_group_code"] = item["code"]
        else: params["query"] = item["q"]
        
        res = requests.get(url, headers=headers, params=params).json()
        for d in res.get('documents', []):
            if item.get("name") == "병원" and any(w in d['place_name'] for w in ['피부과', '동물', '요양', '한방']):
                continue
            folium.Marker([float(d['y']), float(d['x'])], tooltip=f"{item.get('name', item.get('q'))}: {d['place_name']}",
                          icon=folium.Icon(color=item['color'], icon=item['icon'])).add_to(m)

    ### html로 저장 
    clean_addr = re.sub(r'[\\/:*?"<>|]', '', target_address).replace(' ', '_')
    file_name = f"{clean_addr}_분석지도.html"

    m.save(file_name)

# ==========================================
# 3. 프로그램 실행
# ==========================================
if __name__ == "__main__":
    user_input = input("분석할 주소를 입력하세요(서구,영도구만 검색가능): ")
    if user_input.strip():
        run_enhanced_analysis(user_input)

분석할 주소를 입력하세요(서구,영도구만 검색가능):  부산광역시 서구 해돋이로153번길 9-17



🔍 [부산광역시 서구 해돋이로153번길 9-17] 입지 분석을 시작합니다...
🎉 분석 완료! 지도가 '부산광역시_서구_해돋이로153번길_9-17_분석지도.html'로 저장되었습니다.



In [67]:
import pandas as pd
import requests
import folium
from math import radians, cos, sin, asin, sqrt
import re
import os

# ==========================================
# 1. 거리 계산 함수
# ==========================================
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 
    return c * r * 1000

# ==========================================
# 2. 메인 분석 및 파일 생성 함수
# ==========================================
def run_enhanced_analysis(target_address):
    api_key = "40641746de52a507f78f6c31be11e9c9"
    headers = {"Authorization": f"KakaoAK {api_key}"}


    stats = {
        "분석 대상 주소": target_address,
        "500m 내 버스정류장 수": 0,
        "500m 내 지하철역 수": 0,
        "500m 내 요양시설 수": 0,
        "500m 내 병원 수": 0,
        "500m 내 약국 수": 0,
        "가장 가까운 종합병원 거리": "정보 없음",
        "가장 가까운 대학병원 거리": "정보 없음"
    }

    geo_res = requests.get("https://dapi.kakao.com/v2/local/search/address.json", headers=headers, params={"query": target_address}).json()
    if not geo_res.get('documents'): 
        print("❌ 주소를 찾을 수 없습니다.")
        return
    
    c_lat, c_lng = float(geo_res['documents'][0]['y']), float(geo_res['documents'][0]['x'])

    h_info_html = []
    hospital_types = ["종합병원", "대학병원"]
    for q in hospital_types:
        res = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", headers=headers, 
                           params={"query": q, "category_group_code": "HP8", "x": c_lng, "y": c_lat, "radius": 15000, "sort": "distance"}).json()
        found_html = "정보 없음"
        for d in res.get('documents', []):
            if all(w not in d['place_name'] for w in ['동물', '요양', '한방', '치과', '피부과']):
                dist = int(d['distance'])
                dist_str = f"{dist/1000:.1f}km" if dist >= 1000 else f"{dist}m"
                found_html = f"<b>{d['place_name']}</b> ({dist_str})"
                
                if q == "종합병원": stats["가장 가까운 종합병원 거리"] = f"{d['place_name']} ({dist_str})"
                else: stats["가장 가까운 대학병원 거리"] = f"{d['place_name']} ({dist_str})"
                break
        h_info_html.append(found_html)

    m = folium.Map(location=[c_lat, c_lng], zoom_start=16, tiles='CartoDB positron')
    folium.Circle(location=[c_lat, c_lng], radius=500, color='#3498db', fill=True, fill_opacity=0.05, weight=2).add_to(m)
    
    # 🌟 타겟 마커 말풍선 크기/글씨 확대
    center_tooltip = f"""
    <div style='width:300px; font-size:14px; line-height:1.6; word-break:keep-all; white-space:normal;'>
        <b>[분석 대상지]</b><br>
        {target_address}<hr style='margin: 5px 0;'>
        🚨 종합병원: {h_info_html[0]}<br>
        🏥 대학병원: {h_info_html[1]}
    </div>
    """
    folium.CircleMarker([c_lat, c_lng], radius=20, color='red', fill=True, fill_color='red', fill_opacity=0.4, z_index=1000).add_to(m)
    folium.Marker([c_lat, c_lng], icon=folium.Icon(color='black', icon='star', prefix='fa'), tooltip=center_tooltip, z_index=1001).add_to(m)

    try:
        # A. 시내버스 (글씨 확대)
        bus_df = pd.read_csv('busan_bus_stops.csv', encoding='utf-8-sig')
        for _, row in bus_df.iterrows():
            if haversine(c_lng, c_lat, row['경도'], row['위도']) <= 500:
                stats["500m 내 버스정류장 수"] += 1
                bus_tooltip = f"<div style='font-size:14px;'>🚌 시내버스: {row['정류장명']}</div>"
                folium.Marker([row['위도'], row['경도']], tooltip=bus_tooltip, icon=folium.Icon(color='orange', icon='flag')).add_to(m)

        # B. 요양시설 (말풍선 너비 확장 및 자동 줄바꿈 적용)
        care_df = pd.concat([pd.read_csv('seogu_care_centers_crawled_final.csv'), 
                             pd.read_csv('yeongdogu_care_centers_crawled_final.csv')], ignore_index=True)
        care_df['위도'] = pd.to_numeric(care_df['위도'], errors='coerce')
        care_df['경도'] = pd.to_numeric(care_df['경도'], errors='coerce')
        care_df = care_df.dropna(subset=['위도', '경도', '장기요양기관', '주소'])

        care_grouped = care_df.groupby(['장기요양기관', '주소', '위도', '경도'], as_index=False).agg({
            '급여종류': lambda x: ', '.join(sorted(set(x.dropna().astype(str)))), 
            '전화번호': 'first',
            '정원': 'first',
            '현원': 'first'
        })
        
        for _, row in care_grouped.iterrows():
            if haversine(c_lng, c_lat, row['경도'], row['위도']) <= 500:
                stats["500m 내 요양시설 수"] += 1
                
                # 🌟 요양시설 말풍선 최적화: 폭을 320px로 넓히고, 글자 크기를 14px로 키우며, 긴 주소가 잘 넘어오도록 설정
                care_tooltip = f"""
                <div style='width:320px; font-size:14px; line-height:1.6; word-break:keep-all; white-space:normal;'>
                    <b style='font-size:15px;'>🏥 {row['장기요양기관']}</b><br>
                    🏷️ 제공서비스: <span style='color:blue;'>{row['급여종류']}</span><br>
                    📍 주소: {row['주소']}<br>
                    📞 전화: {row.get('전화번호', '정보없음')}<br>
                    👥 정원: {row.get('정원', '-')}명 / 현원: {row.get('현원', '-')}명
                </div>
                """
                folium.Marker([row['위도'], row['경도']], tooltip=care_tooltip, icon=folium.Icon(color='purple', icon='heart')).add_to(m)
    except Exception as e:
        print(f"⚠️ 로컬 파일 처리 에러: {e}")

    # [5] API 데이터 연동 (글씨 확대)
    infra_list = [
        {"code": "HP8", "name": "병원", "color": "red", "icon": "plus"},
        {"code": "PM9", "name": "약국", "color": "green", "icon": "plus"},
        {"code": "SW8", "name": "지하철역", "color": "blue", "icon": "info-sign"}
    ]
    
    for item in infra_list:
        url = "https://dapi.kakao.com/v2/local/search/category.json"
        params = {"category_group_code": item["code"], "x": c_lng, "y": c_lat, "radius": 500, "sort": "distance", "size": 15}
        
        res = requests.get(url, headers=headers, params=params).json()
        for d in res.get('documents', []):
            if item["name"] == "병원" and any(w in d['place_name'] for w in ['피부과', '동물', '요양', '한방']):
                continue
            
            if item["name"] == "병원": stats["500m 내 병원 수"] += 1
            elif item["name"] == "약국": stats["500m 내 약국 수"] += 1
            elif item["name"] == "지하철역": stats["500m 내 지하철역 수"] += 1

            # API 인프라 말풍선 최적화
            api_tooltip = f"<div style='font-size:14px;'>{item['name']}: {d['place_name']}</div>"
            folium.Marker([float(d['y']), float(d['x'])], tooltip=api_tooltip, icon=folium.Icon(color=item['color'], icon=item['icon'])).add_to(m)

    ### html과 csv로 저장
    clean_addr = re.sub(r'[\\/:*?"<>|]', '', target_address).replace(' ', '_')
    map_filename = f"{clean_addr}_분석지도.html"
    csv_filename = f"{clean_addr}_분석통계.csv"
    m.save(map_filename)
    stats_df = pd.DataFrame([stats])
    stats_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    print(f"지도: '{map_filename}'\n통계표: '{csv_filename}'")
    print("-" * 40)
    for k, v in stats.items():
        print(f"{k}: {v}")
    print("-" * 40)

# ==========================================
# 3. 프로그램 실행
# ==========================================
if __name__ == "__main__":
    user_input = input("분석할 주소를 입력하세요: ")
    print("")
    if user_input.strip():
        run_enhanced_analysis(user_input)

분석할 주소를 입력하세요:  부산광역시 서구 구덕로 322번길 63



지도: '부산광역시_서구_구덕로_322번길_63_분석지도.html'
통계표: '부산광역시_서구_구덕로_322번길_63_분석통계.csv'
----------------------------------------
분석 대상 주소: 부산광역시 서구 구덕로 322번길 63
500m 내 버스정류장 수: 45
500m 내 지하철역 수: 1
500m 내 요양시설 수: 12
500m 내 병원 수: 13
500m 내 약국 수: 7
가장 가까운 종합병원 거리: 동아대학교병원 (720m)
가장 가까운 대학병원 거리: 동아대학교 대신병원 (700m)
----------------------------------------


In [66]:
import pandas as pd

file_name = '부산광역시_서구_구덕로_322번길_63_분석통계.csv'


df = pd.read_csv(file_name, encoding='utf-8')
df

,분석 대상 주소,500m 내 버스정류장 수,500m 내 지하철역 수,500m 내 요양시설 수,500m 내 병원 수,500m 내 약국 수,가장 가까운 종합병원 거리,가장 가까운 대학병원 거리
0,부산광역시 서구 구덕로 322번길 63,45,1,12,13,7,동아대학교병원 (720m),동아대학교 대신병원 (700m)
